# Gramática no sampler: um exemplo simples

A cada passo, o modelo dá uma **nota** para cada token do vocabulário e o sampler **sorteia** um.
A gramática funciona como uma **trava de teclado**: só os tokens que mantêm o código válido podem ser sorteados.

Este notebook mostra isso em 3 partes:
1. o código visto como tokens;
2. a trava funcionando passo a passo;
3. um "macaco no teclado" (modelo aleatório) gerando código **sem** e **com** a trava.

In [1]:
import ast
import numpy as np

from codelm.tokenizer import load_tokenizer, encode_code, encode_segments
from codelm.indentation import encode_indentation
from codelm.grammar import GrammarConstraint

tok = load_tokenizer()
print("tamanho do vocabulário:", tok.get_vocab_size())

tamanho do vocabulário: 16384


## 1. Código é uma sequência de tokens

O modelo nunca vê o texto, só esta lista. (`Ġ` é um espaço e `Ċ` é uma quebra de linha;
a indentação vira `<|indent|>`/`<|dedent|>`.)

In [2]:
codigo = "def soma(a, b):\n    return a + b\n"
ids = encode_code(tok, codigo, "python")
print([tok.id_to_token(i) for i in ids])

['def', 'Ġsom', 'a', '(', 'a', ',', 'Ġb', ')', ':', 'Ċ', '<|indent|>', '<|indent|>', 'return', 'Ġa', 'Ġ+', 'Ġb', 'Ċ', '<|dedent|>', '<|dedent|>']


## 2. A trava em alguns pontos

Damos à gramática um começo de código e perguntamos: **quantos tokens (dos 16.384) ela permite agora?**
Mostramos também os permitidos que não são pedaços de nome (nomes são quase todo o vocabulário).

In [3]:
def texto_do_token(token):
    return tok.decode([token], skip_special_tokens=False)

def trava_depois_de(comeco):
    trava = GrammarConstraint("python")
    # Codifica só o começo (encode_code completaria com quebra de linha e dedents finais).
    ids = encode_segments(tok, encode_indentation(comeco, "python"))
    while tok.id_to_token(ids[-1]) == "<|dedent|>":
        ids.pop()
    for token in ids:
        trava.consume(token)
    mascara = trava.allowed()
    exemplos = [texto_do_token(i) for i in np.flatnonzero(mascara)]
    exemplos = [t for t in exemplos if t.strip() and not t.strip().isalnum() and "�" not in t][:10]
    print(f"{comeco!r:24} → {mascara.sum():>6,} permitidos   ex.: {exemplos}")

trava_depois_de("x = ")
trava_depois_de("def soma(a, b)")
trava_depois_de("if x == 1:\n")
trava_depois_de("y = (1 + 2")
trava_depois_de("print('oi'")

# o fim do programa está permitido?
for comeco in ["y = (1 + 2", "y = (1 + 2)\n"]:
    trava = GrammarConstraint("python")
    for token in encode_segments(tok, encode_indentation(comeco, "python")):
        trava.consume(token)
    print(f"{comeco!r:24} → <|endoftext|> permitido? {bool(trava.allowed()[tok.token_to_id('<|endoftext|>')])}")

'x = '                   → 15,978 permitidos   ex.: ['"', "'", '(', '*', '+', '-', '.', '[', '\\', '_']
'def soma(a, b)'         →     41 permitidos   ex.: ['-', ':', '\\', ' -', '->', ' :', ' ->', ' \\']
'if x == 1:\n'           →     95 permitidos   ex.: ['<|indent|>', '#', ' #']
'y = (1 + 2'             → 15,465 permitidos   ex.: ['!', '#', '%', '&', '(', ')', '*', '+', ',', '-']
"print('oi'"             → 15,909 permitidos   ex.: ['!', '"', '#', '%', '&', "'", '(', ')', '*', '+']
'y = (1 + 2'             → <|endoftext|> permitido? False
'y = (1 + 2)\n'          → <|endoftext|> permitido? True


- Depois de `x = ` cabe qualquer expressão (qualquer nome, número, `(`, `[`...), então quase tudo é permitido.
- Depois de `def soma(a, b)` só sobram uns 40: `:`, `->` e espaços. Os outros 16 mil estão **travados**.
- Depois de `if x == 1:` + quebra de linha, o bloco **tem** que começar: só `<|indent|>` (ou linha em branco/comentário).
- Com um parêntese aberto, quase tudo continua permitido (continuar a conta, fechar com `)`, até quebrar a linha),
  mas `<|endoftext|>` fica travado: o programa não pode acabar com o parêntese aberto.

## 3. O macaco no teclado

Como ainda não temos modelo treinado, usamos uma função que dá **notas aleatórias** para todos os tokens.
Ela tem um empurrãozinho para alguns tokens, senão nunca terminaria: quebra de linha, `<|dedent|>`,
os que fecham coisas (`)`, `]`, `}`, aspas) e `<|endoftext|>` (fim, só depois de 15 tokens).
Sem esse empurrão, com a trava ligada o macaco escreveria um nome gigante para sempre:
há uns 13 mil pedaços de palavra no vocabulário, todos válidos no meio de um nome.

O sorteio é sempre o mesmo; a única diferença entre os dois testes é se a trava está ligada.

In [4]:
rng = np.random.default_rng(0)
FIM = tok.token_to_id("<|endoftext|>")

QUEBRA = tok.token_to_id("Ċ")
DEDENT = tok.token_to_id("<|dedent|>")
FECHA = [tok.token_to_id(t) for t in [")", "]", "}", '"', "'"]]

def macaco(ids_ate_agora):
    """Faz o papel do modelo: uma nota aleatória para cada token (mais o empurrãozinho)."""
    notas = rng.normal(size=tok.get_vocab_size()) * 2
    notas[FIM] += 12 if len(ids_ate_agora) >= 15 else -20
    notas[QUEBRA] += 11
    notas[DEDENT] += 10
    notas[FECHA] += 10
    return notas

def sorteia(notas, mascara=None):
    if mascara is not None:
        notas = np.where(mascara, notas, -np.inf)   # a trava: inválidos nunca são sorteados
    probs = np.exp(notas - notas.max())
    return int(rng.choice(len(notas), p=probs / probs.sum()))

def gera(com_trava, max_tokens=200):
    trava = GrammarConstraint("python") if com_trava else None
    ids = []
    for _ in range(max_tokens):
        mascara = trava.allowed() if trava else None
        token = sorteia(macaco(ids), mascara)
        if trava:
            trava.consume(token)
        if token == FIM:
            return ids, True
        ids.append(token)
    return ids, False

def e_python_valido(texto):
    try:
        ast.parse(texto)
        return True
    except SyntaxError:
        return False

### Sem a trava: o macaco escreve qualquer coisa

In [5]:
from codelm.tokenizer import decode_code

def testa(com_trava, vezes=40, mostrar=3):
    validos = terminados = 0
    for _ in range(vezes):
        ids, terminou = gera(com_trava)
        if not terminou:
            continue   # chegou no limite de tokens antes de terminar
        terminados += 1
        texto = decode_code(tok, ids, "python")
        validos += e_python_valido(texto)
        if terminados <= mostrar:
            print(f"válido: {e_python_valido(texto)!s:5} | {texto[:100]!r}")
    print(f"\n{terminados} programas terminados, {validos} são Python válido")

testa(com_trava=False)

válido: False | ' trabal}]}dictvere"\nanken\'\'")\n"'
válido: False | ']}]"\')]])"\']) sue)_json}\n ignored\'\'\'\n'
válido: False | '\n\n")\nprovides)")\n})"\n'



40 programas terminados, 0 são Python válido


### Com a trava: o macaco continua aleatório, mas só pode apertar teclas válidas

O resultado parece sem sentido (strings cheias de símbolos, nomes soltos), porque o macaco não entende nada.
Muitas vezes ele também não termina dentro do limite de tokens (fica preso dentro de uma string ou parêntese).
Mas **é sempre Python sintaticamente válido**. Um modelo treinado dá notas que fazem sentido;
a trava só garante que a sintaxe nunca quebra.

In [6]:
testa(com_trava=True)

válido: True  | 'Filter\n\n\n\'}\'\n\' loginurface death]\'\'\'")"\n'


válido: True  | '\')}"} KEY]\'\nSite\n\n"]}}"\n'
válido: True  | '\'}]""}]}cir}}]}] unwrap} spécialpage}\'\n'



9 programas terminados, 9 são Python válido


## Resumo

| | quem decide |
|---|---|
| **o que** escrever (nomes, lógica) | o modelo (aqui, o macaco) |
| **se a sintaxe é válida** | a gramática (`codelm/grammars/python.lark`), aplicada pelo llguidance |

Na geração de verdade, `codelm.sampler.generate` faz exatamente este laço, com o modelo treinado
no lugar do `macaco` e sorteio com temperatura, top-k e top-p.